# Using Project Source Code

This chapter demonstrates how to import library code from `src/` without
`sys.path` hacks. The project is installed in editable mode when you run
`uv sync`, so notebooks can import packages directly.

The example pipeline:

1. Generate synthetic data with a small simulation
2. Bundle a Random Forest and an SVM (with `StandardScaler`) into a single `Pipeline` whose outer step is the shared `FeatureUnion` and whose inner step is a `VotingRegressor` (no meta-learner / `final_estimator` needed)
3. Hand that composite to `compare_models`, which fits each base, computes bootstrap confidence intervals, MAPIE split conformal prediction intervals, and bootstrap-CI'd regression metrics
4. Visualize the comparison with faceted Altair charts

## Imports

No path bootstrapping is required — the packages are installed by `uv sync`.

In [31]:
from sklearn.ensemble import VotingRegressor
from sklearn.pipeline import Pipeline

from analysis import compare_models
from core import ModelKind, Settings, build_split_dataset, expand_features
from prediction import (
    ENSEMBLE_STEP,
    FEATURES_STEP,
    random_forest_regressor,
    svm_regressor,
)
from simulation import generate_dataset
from visualization import (
    plot_dataset,
    plot_interval_metrics,
    plot_intervals,
    plot_regression_metrics,
)

## Generate synthetic data

In [32]:
settings = Settings(n_samples=500, seed=0, svm_gamma=0.1)
data = generate_dataset(settings)
data.head()

,x,y
0,2.739234,9.283791
1,-4.604266,-12.661366
2,-9.180530,-13.440572
3,-9.669447,-11.312229
4,6.265405,-1.103287


In [33]:
plot_dataset(data)

alt.LayerChart(...)

## Split into train, calibration, and test sets

In [34]:
split_data = build_split_dataset(data, random_state=settings.seed)
split_data["split"].value_counts()

split
training       350
calibration     75
evaluation      75
Name: count, dtype: int64

## Build the composite estimator

The polynomial + Fourier feature expansion is identical across regressors, so we build it
**once** with `expand_features(settings)` and place it as the **outer** step of a
`sklearn.pipeline.Pipeline`. The inner step is a `sklearn.ensemble.VotingRegressor` whose
`estimators` list holds the per-model bases — no `final_estimator` is needed, since voting
just averages base predictions. The SVM factory additionally wraps `SVR` behind a
`StandardScaler` (an inner pipeline), so scaling happens after the shared feature expansion
and only for the model that needs it.

This shape renders as a single `FeatureUnion` at the top branching into the two regressors —
no per-model duplication of the feature stage. `compare_models` introspects
`regressors.named_steps[ENSEMBLE_STEP].estimators` to reconstruct one `regression_pipeline(...)`
per base for fitting, bootstrap, and conformal calibration.

In [35]:
features = expand_features(settings)
features

,"transformer_list transformer_list: list of (str, transformer) tuplesList of transformer objects to be applied to the data. The firsthalf of each tuple is the name of the transformer. The transformer canbe 'drop' for it to be ignored or can be 'passthrough' for features tobe passed unchanged... versionadded:: 1.1 Added the option `""passthrough""`... versionchanged:: 0.22 Deprecated `None` as a transformer in favor of 'drop'.","[('polynomialfeatures', ...), ('fourierfeatures', ...)]"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer.Keys are transformer names, values the weights.Raises ValueError if key not present in ``transformer_list``.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, default=TrueIf True, :meth:`get_feature_names_out` will prefix all feature nameswith the name of the transformer that generated that feature.If False, :meth:`get_feature_names_out` will not prefix any featurenames and will error if feature names are not unique... versionadded:: 1.5",True
,"degree degree: int or tuple (min_degree, max_degree), default=2If a single int is given, it specifies the maximal degree of thepolynomial features. If a tuple `(min_degree, max_degree)` is passed,then `min_degree` is the minimum and `max_degree` is the maximumpolynomial degree of the generated features. Note that `min_degree=0`and `min_degree=1` are equivalent as outputting the degree zero term isdetermined by `include_bias`.",5
,"include_bias include_bias: bool, default=TrueIf `True` (default), then include a bias column, the feature in whichall polynomial powers are zero (i.e. a column of ones - acts as anintercept term in a linear model).",False
,"interaction_only interaction_only: bool, default=FalseIf `True`, only interaction features are produced: features that areproducts of at most `degree` *distinct* input features, i.e. terms withpower of 2 or higher of the same input feature are excluded:- included: `x[0]`, `x[1]`, `x[0] * x[1]`, etc.- excluded: `x[0] ** 2`, `x[0] ** 2 * x[1]`, etc.",False
,"order order: {'C', 'F'}, default='C'Order of output array in the dense case. `'F'` order is faster tocompute, but may slow down subsequent estimators... versionadded:: 0.21",'C'
,n_terms,6
,frequency,0.5


In [36]:
random_forest = random_forest_regressor(settings)
random_forest

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",0
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_le

In [37]:
svm = svm_regressor(settings)
svm

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('svr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.1
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0


In [38]:
regressors = Pipeline(
    steps=[
        (FEATURES_STEP, features),
        (
            ENSEMBLE_STEP,
            VotingRegressor(
                estimators=[
                    (ModelKind.RANDOM_FOREST.value, random_forest),
                    (ModelKind.SVM.value, svm),
                ],
            ),
        ),
    ],
)
regressors

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('features', ...), ('ensemble', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformer_list transformer_list: list of (str, transformer) tuplesList of transformer objects to be applied to the data. The firsthalf of each tuple is the name of the transformer. The transformer canbe 'drop' for it to be ignored or can be 'passthrough' for features tobe passed unchanged... versionadded:: 1.1 Added the option `""passthrough""`... versionchanged:: 0.22 Deprecated `None` as a transformer in favor of 'drop'.","[('polynomialfeatures', ...), ('fourierfeatures', ...)]"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer.Keys are transformer names, values the weights.Raises ValueError if key not present in ``transformer_list``.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, default=TrueIf True, :meth:`get_feature_names_out` will prefix all feature nameswith the name of the transformer that generated that feature.If False, :meth:`get_feature_names_out` will not prefix any featurenames and will error if feature names are not unique... versionadded:: 1.5",True
,"degree degree: int or tuple (min_degree, max_degree), default=2If a single int is given, it specifies the maximal degree of thepolynomial features. If a tuple `(min_degree, max_degree)` is passed,then `min_degree` is the minimum and `max_degree` is the maximumpolynomial degree of the generated features. Note that `min_degree=0`and `min_degree=1` are equivalent as outputting the degree zero term isdetermined by `include_bias`.",5
,"include_bias include_bias: bool, default=TrueIf `True` (default), then include a bias column, the feature in whichall polynomial powers are zero (i.e. a column of ones - acts as anintercept term in a l

## Fit, calibrate, score, and bootstrap in one call

`compare_models` walks each `(name, pipeline)` pair in the composite once and returns a
`ModelComparisonReport` dataclass with six tagged DataFrames:

- `predictions` — per-model point predictions with ground truth
- `confidence` — bootstrap confidence intervals (refit-on-resample) per model
- `prediction` — MAPIE split conformal prediction intervals per model
- `regression_metrics` — RMSE/MAE/R² with bootstrap CIs, per model
- `confidence_metrics` and `prediction_metrics` — interval width and MWI scores, per model

All concatenation and `model`-column tagging happens inside `src/`; the notebook stays declarative.

In [39]:
report = compare_models(split_data, regressors, settings)
report.predictions.head()

,x,y_pred,y_true,model
0,2.725673,6.138810,4.347060,random_forest
1,-0.664330,8.191209,7.711474,random_forest
2,0.520445,9.820025,11.161412,random_forest
3,-6.055301,-18.097552,-16.631863,random_forest
4,1.838056,8.887239,8.670207,random_forest


## Visualize the comparison

`plot_intervals` renders one small-multiples panel per model: the data scatter,
the bootstrap confidence band, the conformal prediction band, and the regression line.

In [40]:
plot_intervals(data, report)

alt.HConcatChart(...)

## Pipeline evaluation

`plot_regression_metrics` facets by metric so each metric (RMSE, MAE, R²) gets its own y-axis —
comparing models across metrics on a shared scale would be misleading because the metrics live in
different units. Models sit on the x-axis within each facet, and each error bar shows the
bootstrap CI.

`plot_interval_metrics` facets by interval kind (confidence vs. prediction). Within each facet,
the metrics (width, MWI) are grouped on the x-axis and the models are color-coded — so the chart
compares confidence-against-confidence and prediction-against-prediction, not CI-against-PI.

In [41]:
plot_regression_metrics(report)

alt.FacetChart(...)

In [42]:
plot_interval_metrics(report)

alt.VConcatChart(...)

In [43]:
report.regression_metrics

,metric,lower,upper,model
0,rmse,2.336183,3.721747,random_forest
1,rmse,3.125137,4.251892,svm
2,mae,1.576535,2.511128,random_forest
3,mae,2.468701,3.396562,svm
4,r2,0.873687,0.968196,random_forest
5,r2,0.837791,0.927516,svm


In [44]:
report.confidence_metrics

,kind,metric,value,model
0,confidence,width,1.300702,random_forest
1,confidence,width,1.407690,svm
2,confidence,coverage,0.440000,random_forest
3,confidence,coverage,0.253333,svm


In [45]:
report.prediction_metrics

,kind,metric,value,model
0,prediction,width,16.229529,random_forest
1,prediction,width,20.337669,svm
2,prediction,mwi,18.401142,random_forest
3,prediction,mwi,20.337669,svm
4,prediction,coverage,0.973333,random_forest
5,prediction,coverage,1.000000,svm


## Tests

Each module under `src/` has a matching test module under `tests/`.
Run the full suite with:

```bash
uv run poe test
```

CI runs tests before building the book (`uv run poe ci`).